# 01 · Bronze — dado cru, com rastreabilidade

```
landing (arquivos crus) ──► BRONZE (Iceberg) ──► Silver ──► Gold ─┬─► Postgres ──► Metabase
                                                                  └─► chunks ──► pgvector (próximo módulo)
```

**Regra da Bronze:** guardar como veio. Nada de limpar aqui. Só acrescentamos 3 colunas para saber *de onde* e *quando* veio:
`_ingest_ts`, `_source_file`, `_batch_id`.

**Antes de começar:** `docker compose up -d` (pasta `docker/`) e `pip install -r requirements.txt`.

## Passo 0 · O ambiente está de pé?

In [ ]:
import sys, subprocess
r = subprocess.run([sys.executable, "../scripts/verifica_ambiente.py"],
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr)

## Passo 0.5 · Baixar as dependências do Spark (só na primeira vez)

O Spark precisa de alguns jars (Iceberg, S3A com o AWS SDK, JDBC). Na **primeira execução** ele os baixa da internet, e isso pode levar **vários minutos**, principalmente com a rede da sala dividida. Os arquivos ficam guardados em cache (pasta `.ivy2`) e, depois disso, o Spark abre rápido.

Rode a célula abaixo **em casa, antes da aula**. Ela abre um processo separado, baixa os jars e termina, sem deixar o Spark aberto neste notebook.

In [ ]:
import sys, subprocess

r = subprocess.run([sys.executable, "../scripts/preparar_jars.py"],
                   capture_output=True, text=True)
print(r.stdout[-1500:])      # fim da saída: deve terminar com "Jars baixados e prontos"
print(r.stderr[-1500:])      # se houver erro, a causa aparece aqui

## Passo 1 · Abrir o Spark (modo local)

O Spark roda **dentro deste Python**, com `local[2]` (2 núcleos) e 2 GB. Na primeira vez ele baixa os jars (Iceberg, S3A, JDBC). O S3A traz junto o AWS SDK, que é grande, então pode levar vários minutos e precisa de internet. Os jars ficam em cache em ~/.ivy2 e nas próximas execuções é rápido.
Para mais folga, `SPARK_CORES` e `SPARK_MEMORY` (variáveis de ambiente antes de abrir o Jupyter).
Spark UI: <http://localhost:4040> enquanto houver um job rodando.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))          # para importar a pasta common/
from common import lake_local as L #, medalhao as M usaremos depois
from pyspark.sql import functions as F
spark = L.get_spark()                              # PySpark local: sem cluster
print("Spark", spark.version, "| master:", spark.sparkContext.master, "| storage:", L.STORAGE)

## Passo 2 · Os arquivos chegam na landing (object storage)

In [ ]:
n = L.upload_to_landing(L.DATA_DIR / "lote1", "mercadonordeste/lote1")
print(n, "arquivos enviados")
for chave, tam in L.list_landing("mercadonordeste/lote1"):
    print(f"{tam:>10,} bytes  {chave}")

Abra o console do MinIO (<http://localhost:9001>, `admin` / `admin12345`) e veja o bucket **landing**.
É um *object storage*: não há pastas de verdade, só chaves com `/` no nome. 
Você achou os objetos no bucket warehouse do MinIO, em lakehouse/bronze/<tabela>/data e metadata.

## Passo 3 · Landing → Bronze

In [ ]:
contagens = M.ingest_lote(spark, "lote1", L.landing_uri)
contagens

In [ ]:
spark.sql("SHOW TABLES IN lake.bronze").show()
spark.table("lake.bronze.pedidos").printSchema()
spark.table("lake.bronze.pedidos").select("pedido_id","cliente_id","status","itens","_batch_id","_source_file").show(3, truncate=60)

Repare: `itens` chegou como **lista de structs** (o JSON foi preservado) e os CSVs chegaram como texto. A tipagem é problema da Silver.

## Passo 4 · Idempotência
Rode o Passo 3 de novo. As contagens **não** dobram: a ingestão apaga o lote e regrava. Pipeline que duplica ao reexecutar é pipeline frágil.

In [ ]:
antes = spark.table("lake.bronze.pedidos").count()
M.ingest_lote(spark, "lote1", L.landing_uri)
depois = spark.table("lake.bronze.pedidos").count()
print(antes, "->", depois, "| idempotente:", antes == depois)

## Checkpoint
- [ ] `lake.bronze` tem `clientes`, `produtos`, `pedidos`, `avaliacoes`.
- [ ] Reexecutar não duplica.
- [ ] Você achou os objetos no bucket `warehouse` do MinIO (pastas `bronze/…/data` e `metadata`).

➡️ Próximo: `02_silver.ipynb`